# 03 — Disagreement Exploration

This notebook explores *why* translation services disagree using the four-category typology from `explore_disagreements.py`:

| Category | Description |
|---|---|
| **MEASUREMENT_ARTEFACT** | Disagreement dissolves after normalization (Esperanto: one letter of grammatical agreement) |
| **STRUCTURAL_ABSENCE** | No community equivalent — model borrows or signals absence (Swahili) |
| **PRODUCTIVE_DISAGREEMENT** | Multiple legitimate in-language alternatives reflecting real debate (Arabic) |
| **TRANSMOGRIFICATION** | Confident fluent output untethered from community practice (Gothic) |

Input: `translated_terms/digital_humanities/evaluation/disagreement_analysis.csv`

In [1]:
import os
import sys
import ast
import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable('vegafusion')

sys.path.insert(0, str(Path('..').resolve()))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family
from scripts.exploration.explore_confidence_within_variant import load_variant_df
from scripts.exploration.explore_disagreements import run_disagreement_analysis, CATEGORIES

DATA_DIR   = get_data_directory_path()
TERM       = 'Digital Humanities'
TERM_SLUG  = TERM.lower().replace(' ', '_')
EVAL_DIR   = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')

CATEGORY_ORDER = [
    'STRUCTURAL_ABSENCE', 'TRANSMOGRIFICATION',
    'PRODUCTIVE_DISAGREEMENT', 'MEASUREMENT_ARTEFACT', 'NOT_APPLICABLE'
]
CATEGORY_COLOURS = {
    'STRUCTURAL_ABSENCE':      '#d62728',
    'TRANSMOGRIFICATION':      '#ff7f0e',
    'PRODUCTIVE_DISAGREEMENT': '#2ca02c',
    'MEASUREMENT_ARTEFACT':    '#1f77b4',
    'NOT_APPLICABLE':          '#aec7e8',
}
CATEGORY_LABELS = {
    'STRUCTURAL_ABSENCE':      'Structural Absence',
    'TRANSMOGRIFICATION':      'Transmogrification',
    'PRODUCTIVE_DISAGREEMENT': 'Productive Disagreement',
    'MEASUREMENT_ARTEFACT':    'Measurement Artefact',
    'NOT_APPLICABLE':          'All Agree',
}

## 3.1 Load Data

In [ ]:
analysis_path = os.path.join(EVAL_DIR, 'disagreement_analysis.csv')
if not os.path.exists(analysis_path):
    print('Running explore_disagreements.py...')
    df = run_disagreement_analysis(DATA_DIR, [TERM])
else:
    df = read_csv_file(analysis_path)

df['category_label'] = df['category'].map(CATEGORY_LABELS)

# Load the minimal variant for full translation + rationale access
variant_df = load_variant_df(DATA_DIR, TERM_SLUG, 'minimal')

print(f'{len(df)} languages | {df["category"].value_counts().to_dict()}')
df.head(3)

## 3.2 Category Distribution

In [3]:
counts = (
    df.groupby('category')
    .size()
    .reset_index(name='n')
    .assign(
        label=lambda d: d['category'].map(CATEGORY_LABELS),
        pct=lambda d: (d['n'] / d['n'].sum() * 100).round(1),
        pct_label=lambda d: d.apply(lambda r: f"{r['label']}\n{r['n']} ({r['pct']}%)", axis=1),
    )
)

donut = alt.Chart(counts).mark_arc(innerRadius=60, outerRadius=120).encode(
    theta=alt.Theta('n:Q'),
    color=alt.Color(
        'category:N',
        scale=alt.Scale(
            domain=list(CATEGORY_COLOURS.keys()),
            range=list(CATEGORY_COLOURS.values())
        ),
        legend=alt.Legend(title='Category', labelExpr=
            "{'STRUCTURAL_ABSENCE':'Structural Absence','TRANSMOGRIFICATION':'Transmogrification',"
            "'PRODUCTIVE_DISAGREEMENT':'Productive Disagreement','MEASUREMENT_ARTEFACT':'Measurement Artefact',"
            "'NOT_APPLICABLE':'All Agree'}[datum.label]"
        )
    ),
    tooltip=['label:N', 'n:Q', 'pct:Q']
).properties(width=300, height=300, title='Disagreement Category Distribution')

bar = alt.Chart(counts).mark_bar().encode(
    x=alt.X('n:Q', title='Languages'),
    y=alt.Y('category:N', sort='-x', title=None,
            axis=alt.Axis(labelExpr=
                "{'STRUCTURAL_ABSENCE':'Structural Absence','TRANSMOGRIFICATION':'Transmogrification',"
                "'PRODUCTIVE_DISAGREEMENT':'Productive Disagreement','MEASUREMENT_ARTEFACT':'Measurement Artefact',"
                "'NOT_APPLICABLE':'All Agree'}[datum.label]"
            )),
    color=alt.Color('category:N',
        scale=alt.Scale(domain=list(CATEGORY_COLOURS.keys()), range=list(CATEGORY_COLOURS.values())),
        legend=None),
    text=alt.Text('pct:Q', format='.1f'),
    tooltip=['label:N', 'n:Q', 'pct:Q']
).mark_bar() + alt.Chart(counts).mark_text(align='left', dx=4).encode(
    x=alt.X('n:Q'),
    y=alt.Y('category:N', sort='-x'),
    text=alt.Text('pct:Q', format='.1f')
)

(donut | bar).resolve_scale(color='independent')

alt.HConcatChart(...)

## 3.3 Category × Language Family

In [4]:
interesting_cats = ['STRUCTURAL_ABSENCE', 'TRANSMOGRIFICATION', 'PRODUCTIVE_DISAGREEMENT', 'MEASUREMENT_ARTEFACT']

family_cat = (
    df[df['category'].isin(interesting_cats)]
    .groupby(['language_family', 'category'])
    .size()
    .reset_index(name='n')
    .assign(label=lambda d: d['category'].map(CATEGORY_LABELS))
)

# Total per family for sorting
family_totals = family_cat.groupby('language_family')['n'].sum().reset_index(name='total')
family_cat = family_cat.merge(family_totals, on='language_family')

alt.Chart(family_cat).mark_rect().encode(
    x=alt.X('label:N', sort=list(CATEGORY_LABELS.values())[:4], title=None),
    y=alt.Y('language_family:N',
            sort=alt.EncodingSortField(field='total', order='descending'),
            title='Language Family'),
    color=alt.Color('n:Q',
        scale=alt.Scale(scheme='orangered'),
        title='Languages'),
    tooltip=['language_family:N', 'label:N', 'n:Q']
).properties(
    width=420, height=400,
    title='Disagreement Categories by Language Family'
)

alt.Chart(...)

## 3.4 Measurement Artefacts

Disagreements that dissolve after normalization — the scoring tool's limit, not a real divergence. These are useful as a calibration check: if the string-matching algorithm calls these "different", that tells us something about what we can and can't claim from exact-match scoring.

In [5]:
artefacts = df[df['category'] == 'MEASUREMENT_ARTEFACT'].copy()
print(f'{len(artefacts)} measurement artefacts')

# Show key columns
display(artefacts[[
    'language_code', 'language_name', 'language_family',
    'n_unique_raw', 'n_unique_normalized', 'max_edit_distance', 'evidence',
    'service_translations'
]].sort_values('max_edit_distance', ascending=False))

8 measurement artefacts


,language_code,language_name,language_family,n_unique_raw,n_unique_normalized,max_edit_distance,evidence,service_translations
87,hak,Hakka Chinese,Sino-Tibetan,3,3,2,Normalized to 3 unique value(s); max edit dist...,"{'OpenAI': '數位人文', 'Claude': '數位人文', 'Gemini':..."
130,lad,Ladino / Judeo-Spanish,Romance,2,2,2,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': 'Humanidades Digitales', 'Claude': ..."
158,mwl,Mirandese,Romance,3,3,2,Normalized to 3 unique value(s); max edit dist...,"{'OpenAI': 'Humanidades Dixitales', 'Claude': ..."
13,ast,Asturian,Romance,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': 'Humanidaes Dixitales', 'Claude': '..."
52,da,Danish,Germanic,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'Google Translate': 'Digitale humaniora', 'Ea..."
97,ia,Interlingua,Romance,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': 'Humanitates Digital', 'Claude': 'H..."
163,nb,Norwegian Bokmål,Germanic,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'Google Translate': 'Digital humaniora', 'Ope..."
171,nn,Norwegian Nynorsk,Germanic,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': 'Digitale humaniora', 'Claude': 'Di..."


In [6]:
# Edit distance distribution for artefacts
artefacts_plot = artefacts.dropna(subset=['max_edit_distance']).copy()
artefacts_plot['max_edit_distance'] = artefacts_plot['max_edit_distance'].astype(int)

alt.Chart(artefacts_plot).mark_bar(color=CATEGORY_COLOURS['MEASUREMENT_ARTEFACT']).encode(
    x=alt.X('max_edit_distance:O', title='Max Edit Distance (normalized)'),
    y=alt.Y('count():Q', title='Languages'),
    tooltip=['max_edit_distance:O', 'count():Q']
).properties(
    width=300, height=200,
    title='Edit Distance Distribution — Measurement Artefacts'
)

alt.Chart(...)

## 3.5 Structural Absence

The concept has no community equivalent. The model signals this through loan words, explicit "no equivalent" rationale text, or near-zero coverage. This is the dominant pattern — and it's a finding about where DH as a practice hasn't reached, not a pipeline failure.

In [7]:
absent = df[df['category'] == 'STRUCTURAL_ABSENCE'].copy()
print(f'{len(absent)} structural absence cases')
print(f'  Loan word detected: {absent["loan_words_found"].sum()}')
print(f'  Absence signals in rationale: {(absent["absence_signals"].str.len() > 0).sum()}')
print(f'  Zero services with data: {(absent["n_services_with_data"] == 0).sum()}')

88 structural absence cases
  Loan word detected: 86
  Absence signals in rationale: 75
  Zero services with data: 0


In [8]:
# Absence sub-signals breakdown
absent_signals = (
    absent.assign(
        has_loan=absent['loan_words_found'],
        has_rationale_signal=absent['absence_signals'].str.len() > 0,
        no_data=absent['n_services_with_data'] == 0,
    )
)

signal_counts = pd.DataFrame({
    'signal': ['Loan word in translation', 'Absence keyword in rationale', 'No service data at all'],
    'n': [
        absent_signals['has_loan'].sum(),
        absent_signals['has_rationale_signal'].sum(),
        absent_signals['no_data'].sum(),
    ]
})

alt.Chart(signal_counts).mark_bar(color=CATEGORY_COLOURS['STRUCTURAL_ABSENCE']).encode(
    x=alt.X('n:Q', title='Languages'),
    y=alt.Y('signal:N', sort='-x', title=None),
    tooltip=['signal:N', 'n:Q']
).properties(width=400, height=160, title='Structural Absence — Evidence Signals')

alt.Chart(...)

In [9]:
# Structural absence by family (stacked: loan word vs rationale signal)
absent_family = (
    absent
    .assign(signal_type=lambda d: d.apply(
        lambda r: 'Loan word' if r['loan_words_found']
        else ('Rationale signal' if len(str(r['absence_signals'])) > 2
              else 'No data / other'),
        axis=1
    ))
    .groupby(['language_family', 'signal_type'])
    .size()
    .reset_index(name='n')
)

family_order = (
    absent_family.groupby('language_family')['n'].sum()
    .sort_values(ascending=False).index.tolist()
)

alt.Chart(absent_family).mark_bar().encode(
    x=alt.X('n:Q', title='Languages'),
    y=alt.Y('language_family:N', sort=family_order, title=None),
    color=alt.Color('signal_type:N',
        scale=alt.Scale(
            domain=['Loan word', 'Rationale signal', 'No data / other'],
            range=['#d62728', '#ff9896', '#ffcdd2']
        ),
        title='Evidence'
    ),
    tooltip=['language_family:N', 'signal_type:N', 'n:Q']
).properties(width=400, height=380, title='Structural Absence by Language Family')

alt.Chart(...)

In [10]:
# Case study: Tagalog (tl) — structural absence via loan word borrowing
# Google Translate returns 'Digital Humanities' unchanged; EasyNMT produces
# 'Digital na mga Tao' (Digital People); shows unadapted borrowing + garbled output
tl_row = df[df['language_code'] == 'tl'].iloc[0] if 'tl' in df['language_code'].values else None
if tl_row is not None:
    print(f'Tagalog (tl) — category: {tl_row["category"]}')
    print(f'Evidence: {tl_row["evidence"]}')
    print(f'Service translations:\n  {tl_row["service_translations"]}')
    print(f'Absence signals: {tl_row["absence_signals"]}')
    if variant_df is not None and 'tl' in variant_df['language_code'].values:
        tl_v = variant_df[variant_df['language_code'] == 'tl'].iloc[0]
        for svc in ['claude', 'openai', 'gemini', 'ollama']:
            rationale = tl_v.get(f'{svc}_translation_rationale', '')
            if rationale and str(rationale) != 'nan':
                print(f'\n{svc.upper()} rationale: {str(rationale)[:400]}')

Tagalog (tl) — category: STRUCTURAL_ABSENCE
Evidence: Loan word detected: True; absence signals: 5; services with data: 8
Service translations:
  {'Google Translate': 'Digital Humanities', 'EasyNMT': 'Digital na mga Tao', 'Lingvanex': 'Digital Humanities', 'OpenAI': 'Diyital na Humanidades', 'Claude': 'Digital na Humanidades', 'Gemini': 'Digital na Humanidades', 'First Ollama': 'Digital na Sining ng Panahon', 'Second Ollama': 'Digital na mga Agham'}
Absence signals: Claude:loanword; OpenAI:retain.*english; Gemini:loanword; Gemini:borrowing; Gemini:transliterat

CLAUDE rationale: This translation best captures the established academic field name. 'Humanidades' is the widely recognized Tagalog/Spanish loanword for 'humanities' used in Philippine academic contexts, making it immediately recognizable to Filipino scholars. 'Digital na' appropriately modifies it following Tagalog grammar patterns. While 'Sining at Agham Panlipunan' (arts and social sciences) could be a purely i

OPENAI ratio

## 3.6 Productive Disagreement

Multiple legitimate in-language alternatives reflecting real scholarly debate. Wikipedia presence is the key signal — it means some community has adopted the term and there is something to disagree *about*. These are the most epistemically interesting cases: the pipeline surfaces debates that a single-output system would have buried.

In [11]:
productive = df[df['category'] == 'PRODUCTIVE_DISAGREEMENT'].copy()
print(f'{len(productive)} productive disagreement cases')
display(productive[[
    'language_code', 'language_name', 'language_family',
    'n_unique_raw', 'has_wikipedia', 'debate_signals', 'service_translations'
]].sort_values('n_unique_raw', ascending=False))

35 productive disagreement cases


,language_code,language_name,language_family,n_unique_raw,has_wikipedia,debate_signals,service_translations
268,zu,Zulu,African (Niger-Congo),8,True,NaN,"{'Wikipedia': 'Ezezibhangqiwe zoLuntu', 'Googl..."
217,sr,Serbian,Slavic,8,True,NaN,"{'Wikipedia': 'Дигитална хуманистика', 'Google..."
223,ta,Tamil,Dravidian,7,True,NaN,"{'Wikipedia': 'எண்ணிம மனிதவியல்', 'Google Tran..."
107,it,Italian,Romance,7,True,NaN,"{'Wikipedia': 'Informatica umanistica', 'Googl..."
64,eu,Basque,Caucasian / Isolate,6,True,NaN,"{'Wikipedia': 'Humanitate digitalak', 'Google ..."
250,vi,Vietnamese,Mainland SE Asian,6,True,NaN,"{'Wikipedia': 'Nhân văn số', 'Google Translate..."
136,lmo,Lombard,Romance,6,True,NaN,"{'Wikipedia': 'Informatega umanistega', 'Googl..."
209,sh,Serbo-Crotian,Slavic,5,True,NaN,"{'Wikipedia': 'Digitalne humanističke nauke', ..."
9,ar,Arabic,Afro-Asiatic,5,True,NaN,"{'Wikipedia': 'إنسانيات رقمية', 'Google Transl..."
109,ja,Japanese,Japonic/Koreanic,4,True,Claude:also used,"{'Wikipedia': 'デジタル・ヒューマニティーズ', 'Google Transl..."


In [12]:
# Productive disagreement: how many unique translations per language?
prod_unique = (
    productive.groupby('n_unique_raw')
    .size()
    .reset_index(name='n_languages')
)

alt.Chart(prod_unique).mark_bar(color=CATEGORY_COLOURS['PRODUCTIVE_DISAGREEMENT']).encode(
    x=alt.X('n_unique_raw:O', title='Unique translation candidates'),
    y=alt.Y('n_languages:Q', title='Languages'),
    tooltip=['n_unique_raw:O', 'n_languages:Q']
).properties(
    width=300, height=200,
    title='Productive Disagreement — Candidate Count'
)

alt.Chart(...)

In [13]:
# Case study: Arabic
ar_row = df[df['language_code'] == 'ar'].iloc[0] if 'ar' in df['language_code'].values else None
if ar_row is not None:
    print(f'Arabic (ar) — category: {ar_row["category"]}')
    print(f'Evidence: {ar_row["evidence"]}')
    print(f'Service translations:\n  {ar_row["service_translations"]}')
    print(f'Debate signals: {ar_row["debate_signals"]}')
    if variant_df is not None and 'ar' in variant_df['language_code'].values:
        ar_v = variant_df[variant_df['language_code'] == 'ar'].iloc[0]
        for svc in ['claude', 'openai', 'gemini', 'ollama']:
            rationale = ar_v.get(f'{svc}_translation_rationale', '')
            if rationale and str(rationale) != 'nan':
                print(f'\n{svc.upper()} rationale: {str(rationale)[:400]}')

Arabic (ar) — category: PRODUCTIVE_DISAGREEMENT
Evidence: Wikipedia translation present; 5 distinct outputs; debate signals: 0
Service translations:
  {'Wikipedia': 'إنسانيات رقمية', 'Google Translate': 'العلوم الإنسانية الرقمية', 'EasyNMT': 'العلوم البشرية', 'Lingvanex': 'العلوم الإنسانية الرقمية', 'OpenAI': 'الإنسانيات الرقمية', 'Claude': 'العلوم الإنسانية الرقمية', 'Gemini': 'العلوم الإنسانية الرقمية', 'First Ollama': 'علم الإنسانية الرقمي', 'Second Ollama': 'الإنسانيات الرقمية'}
Debate signals: nan

CLAUDE rationale: This is the most accurate and widely accepted translation in Arabic academic discourse. 'الإنسانيات' (al-insāniyyāt) correctly captures the concept of 'humanities' as fields of human knowledge and culture, while 'الرقمية' (al-raqmiyya) means 'digital'. This term follows standard Arabic academic terminology patterns with the adjective following the noun. It is also the translation used by major Ara

OPENAI rationale: The term "الإنسانيات الرقمية" effectively combines th

## 3.7 Transmogrification

Confident, fluent, elaborate output untethered from any community practice. No Wikipedia translation, multiple different outputs, rationales are long and constructive (etymology, root words, compound formation). The rationales are the tell — they reveal a model that cannot say "I don't know" and instead reaches for plausible-seeming analogies.

In [14]:
transmog = df[df['category'] == 'TRANSMOGRIFICATION'].copy()
print(f'{len(transmog)} transmogrification cases')
print(f'  With construction signals: {(transmog["construction_signals"].str.len() > 2).sum()}')
print(f'  Mean unique translations: {transmog["n_unique_raw"].mean():.1f}')

131 transmogrification cases
  With construction signals: 101
  Mean unique translations: 5.0


In [15]:
# Construction keyword frequency across transmogrification cases
all_construction_signals = (
    transmog['construction_signals']
    .dropna()
    .str.split('; ')
    .explode()
    .str.strip()
    .loc[lambda s: s.str.len() > 0]
)

# Extract just the keyword part (after the colon)
kw_counts = (
    all_construction_signals
    .str.split(':', n=1).str[-1].str.strip()
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'keyword', 'construction_signals': 'count', 0: 'keyword', 'count': 'n'})
)
# Handle pandas version differences
if 'count' not in kw_counts.columns:
    kw_counts.columns = ['keyword', 'n']

if not kw_counts.empty:
    alt.Chart(kw_counts.head(15)).mark_bar(color=CATEGORY_COLOURS['TRANSMOGRIFICATION']).encode(
        x=alt.X('n:Q', title='Occurrences'),
        y=alt.Y('keyword:N', sort='-x', title=None),
        tooltip=['keyword:N', 'n:Q']
    ).properties(
        width=400, height=300,
        title='Construction Keywords in Transmogrification Rationales'
    )
else:
    print('No construction signals found in this variant — try running with --variant expert_persona')

In [16]:
# Number of unique translations per language (divergence depth)
transmog_unique = transmog.groupby('n_unique_raw').size().reset_index(name='n_languages')

alt.Chart(transmog_unique).mark_bar(color=CATEGORY_COLOURS['TRANSMOGRIFICATION']).encode(
    x=alt.X('n_unique_raw:O', title='Unique translation candidates'),
    y=alt.Y('n_languages:Q', title='Languages'),
    tooltip=['n_unique_raw:O', 'n_languages:Q']
).properties(
    width=300, height=200,
    title='Transmogrification — Candidate Divergence'
)

alt.Chart(...)

In [17]:
# Case study: Gothic (got) — the archetypal transmogrification case
# No Wikipedia, 5 LLM services all produce different elaborate constructions.
# Rationales mention 'loanword' while ALSO constructing novel Gothic terms —
# the model cannot say no and reaches for etymology instead.
got_row = df[df['language_code'] == 'got'].iloc[0] if 'got' in df['language_code'].values else None
if got_row is not None:
    print(f'Gothic (got) — category: {got_row["category"]}')
    print(f'Evidence: {got_row["evidence"]}')
    print(f'Service translations:\n  {got_row["service_translations"]}')
    print(f'Construction signals: {got_row["construction_signals"]}')
    if variant_df is not None and 'got' in variant_df['language_code'].values:
        got_v = variant_df[variant_df['language_code'] == 'got'].iloc[0]
        for svc in ['claude', 'openai', 'gemini', 'ollama']:
            rationale = got_v.get(f'{svc}_translation_rationale', '')
            if rationale and str(rationale) != 'nan':
                print(f'\n{svc.upper()} rationale:\n{str(rationale)[:600]}')

Gothic (got) — category: TRANSMOGRIFICATION
Evidence: No Wikipedia; 5 distinct outputs; absence signals: 3; construction signals: 6
Service translations:
  {'OpenAI': 'Teknodomeiþos', 'Claude': 'digitala manna-kunþi', 'Gemini': '𐌳𐌹𐌲𐌹𐍄𐌰𐌻𐌰 𐌷𐌿𐌼𐌰𐌽𐌹𐍄𐌰𐍄𐌴𐌹𐍃', 'First Ollama': 'Daídis Huministus', 'Second Ollama': 'Kunsthistorikos saƕa wairþja'}
Construction signals: Claude:formation; Claude:morpholog; Gemini:morpholog; Gemini:suffix; First Ollama:suffix; Second Ollama:suffix

CLAUDE rationale:
This translation combines 'fingrageiþo' (genitive plural of 'fingrageiþ', meaning 'finger-reckoning' or 'digital', from 'figgrs' = finger) with 'mannaleisei' (dative plural of 'mannaleisa', meaning 'humanities' or 'human learning', from 'manna' = human + 'laisein' = teaching/learning). The term reflects Gothic word-formation patterns and avoids the anachronistic Latin loanword 'digitalis'. The previous suggestions either use non-Gothic forms ('Dīghtija' appears to mix Old English/Latin) or Latin loanword

## 3.8 Rationale Analysis

Compare rationale characteristics across categories — length, keyword density, construction signal rate. The hypothesis: transmogrification produces longer, more elaborate rationales as a signal of the model constructing rather than recalling.

In [ ]:
# Attach rationale text from variant_df for length analysis
if variant_df is not None:
    rationale_cols = [c for c in variant_df.columns if c.endswith('_translation_rationale')]
    rat_sub = variant_df[['language_code', 'term_source'] + rationale_cols].copy()

    # Mean rationale length across services per language
    rat_sub['mean_rationale_length'] = rat_sub[rationale_cols].apply(
        lambda row: pd.Series([
            len(str(v)) for v in row if v and str(v) != 'nan'
        ]).mean(),
        axis=1
    )

    df_with_rat = df.merge(
        rat_sub[['language_code', 'term_source', 'mean_rationale_length']],
        on=['language_code', 'term_source'],
        how='left'
    )
    print(df_with_rat.groupby('category')['mean_rationale_length'].agg(['mean','median','count']).round(0))
else:
    df_with_rat = df.copy()
    print('No variant data loaded — skipping rationale length analysis')

In [19]:
if 'mean_rationale_length' in df_with_rat.columns:
    rat_plot = df_with_rat.dropna(subset=['mean_rationale_length']).copy()
    rat_plot = rat_plot[rat_plot['category'].isin(['STRUCTURAL_ABSENCE','TRANSMOGRIFICATION','PRODUCTIVE_DISAGREEMENT'])]
    rat_plot['label'] = rat_plot['category'].map(CATEGORY_LABELS)

    alt.Chart(rat_plot).mark_boxplot(extent='min-max').encode(
        x=alt.X('mean_rationale_length:Q', title='Mean rationale length (chars)'),
        y=alt.Y('label:N', sort=None, title=None),
        color=alt.Color('category:N',
            scale=alt.Scale(domain=list(CATEGORY_COLOURS.keys()), range=list(CATEGORY_COLOURS.values())),
            legend=None)
    ).properties(
        width=450, height=200,
        title='Rationale Length by Disagreement Category'
    )

## 3.9 Rationale Similarity by Category

TF-IDF cosine similarity between service rationales, computed with character n-grams (3–5) so the metric is script-agnostic. High similarity means the models converged on the same *reasoning*; low similarity means the models are reaching for different framings even when they agreed on a translation.

Hypothesis: **PRODUCTIVE_DISAGREEMENT** rows should show high similarity (models reason from the same community evidence even while choosing different terms); **TRANSMOGRIFICATION** rows should show low similarity (each model invents its own construction).

In [ ]:
from scripts.exploration.explore_disagreements import compute_rationale_similarity

# Compute similarity for each language using the loaded variant_df rationales
if variant_df is not None:
    rationale_cols_map = {
        'Claude':  'claude_translation_rationale',
        'OpenAI':  'openai_translation_rationale',
        'Gemini':  'gemini_translation_rationale',
        'Ollama':  'ollama_translation_rationale',
    }

    def _row_similarity(vrow):
        rats = {}
        for svc, col in rationale_cols_map.items():
            val = vrow.get(col)
            rats[svc] = str(val) if val and str(val) != 'nan' else None
        return compute_rationale_similarity(rats)

    sim_series = variant_df.apply(_row_similarity, axis=1)
    variant_df_with_sim = variant_df.copy()
    variant_df_with_sim['mean_rationale_similarity'] = sim_series

    # Drop pre-existing column to avoid _x/_y suffix collision on merge
    df_base = df.drop(columns=['mean_rationale_similarity'], errors='ignore')
    df_with_sim = df_base.merge(
        variant_df_with_sim[['language_code', 'term_source', 'mean_rationale_similarity']],
        on=['language_code', 'term_source'],
        how='left'
    )
    print(df_with_sim.groupby('category')['mean_rationale_similarity'].agg(['mean', 'median', 'count']).round(3))
else:
    df_with_sim = df.copy()
    print('No variant data — skipping similarity analysis')

In [21]:
sim_plot = df_with_sim.dropna(subset=['mean_rationale_similarity']).copy()
focus_cats = ['STRUCTURAL_ABSENCE', 'TRANSMOGRIFICATION', 'PRODUCTIVE_DISAGREEMENT']
sim_plot = sim_plot[sim_plot['category'].isin(focus_cats)]
sim_plot['label'] = sim_plot['category'].map(CATEGORY_LABELS)

selection = alt.selection_point(fields=['category'], bind='legend')

_scale = alt.Scale(domain=list(CATEGORY_COLOURS.keys()), range=list(CATEGORY_COLOURS.values()))
_label_expr = (
    "{'STRUCTURAL_ABSENCE':'Structural Absence','TRANSMOGRIFICATION':'Transmogrification',"
    "'PRODUCTIVE_DISAGREEMENT':'Productive Disagreement'}[datum.label]"
)
sel_color = alt.Color('category:N', scale=_scale, title='Category',
    legend=alt.Legend(labelExpr=_label_expr, symbolType='circle',
                      orient='right', titleFontSize=12, labelFontSize=11))

alt.renderers.enable('default')

box = alt.Chart(sim_plot).mark_boxplot(extent='min-max').encode(
    x=alt.X('mean_rationale_similarity:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean pairwise rationale similarity (TF-IDF cosine)'),
    y=alt.Y('label:N', sort=None, title=None),
    color=sel_color,
    # opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.2)),
    tooltip=['label:N', alt.Tooltip('mean_rationale_similarity:Q', format='.3f')],
).properties(
    width=450, height=180,
    title='Rationale Similarity by Disagreement Category'
)

points = alt.Chart(sim_plot).mark_circle(size=60).encode(
    x=alt.X('n_unique_raw:Q', title='Unique translation candidates'),
    y=alt.Y('mean_rationale_similarity:Q', scale=alt.Scale(domain=[0, 1]),
            title='Rationale similarity'),
    color=sel_color,
    # opacity=alt.when(selection).then(alt.value(0.8)).otherwise(alt.value(0.1)),
    tooltip=['language_name:N', 'label:N',
             alt.Tooltip('mean_rationale_similarity:Q', format='.3f'),
             'n_unique_raw:Q'],
).properties(
    width=420, height=250,
    title='Rationale Similarity vs Translation Divergence'
)

box & points

alt.VConcatChart(...)

## 3.10 Full Table — All Disagreement Cases

In [22]:
display_cols = [
    'language_code', 'language_name', 'language_family', 'category',
    'n_unique_raw', 'has_wikipedia', 'loan_words_found',
    'absence_signals', 'construction_signals', 'debate_signals',
    'evidence', 'service_translations'
]

pd.set_option('display.max_colwidth', 80)
display(
    df[display_cols]
    .sort_values(['category', 'language_family', 'language_name'])
    .reset_index(drop=True)
)

,language_code,language_name,language_family,category,n_unique_raw,has_wikipedia,loan_words_found,absence_signals,construction_signals,debate_signals,evidence,service_translations
0,da,Danish,Germanic,MEASUREMENT_ARTEFACT,2,False,True,Claude:borrowed,Claude:compound; Gemini:compound; Gemini:classical; First Ollama:suffix; Fir...,NaN,Normalized to 2 unique value(s); max edit distance 1 ≤ threshold 2,"{'Google Translate': 'Digitale humaniora', 'EasyNMT': 'Digitale humaniora', ..."
1,nb,Norwegian Bokmål,Germanic,MEASUREMENT_ARTEFACT,2,False,True,Claude:borrowed; Claude:loanword,Claude:compound; Gemini:classical; First Ollama:prefix; Second Ollama:prefix,Claude:also used,Normalized to 2 unique value(s); max edit distance 1 ≤ threshold 2,"{'Google Translate': 'Digital humaniora', 'OpenAI': 'Digitale humaniora', 'C..."
2,nn,Norwegian Nynorsk,Germanic,MEASUREMENT_ARTEFACT,2,False,True,NaN,First Ollama:combining; Second Ollama:combining,NaN,Normalized to 2 unique value(s); max edit distance 1 ≤ threshold 2,"{'OpenAI': 'Digitale humaniora', 'Claude': 'Digital humaniora', 'Gemini': 'D..."
3,ast,Asturian,Romance,MEASUREMENT_ARTEFACT,2,False,False,First Ollama:borrowed; Second Ollama:borrowed,NaN,NaN,Normalized to 2 unique value(s); max edit distance 1 ≤ threshold 3,"{'OpenAI': 'Humanidaes Dixitales', 'Claude': 'Humanidaes Dixitales', 'Gemini..."
4,ia,Interlingua,Romance,MEASUREMENT_ARTEFACT,2,False,True,NaN,NaN,NaN,Normalized to 2 unique value(s); max edit distance 1 ≤ threshold 2,"{'OpenAI': 'Humanitates Digital', 'Claude': 'Humanitates Digital', 'Gemini':..."
...,...,...,...,...,...,...,...,...,...,...,...,...
264,uz_AF,Uzbeki Afghanistan,Turkic,TRANSMOGRIFICATION,4,False,False,Gemini:loanword,Claude:compound; Claude:construct,Gemini:variant,No Wikipedia; 4 distinct outputs; absence signals: 1; construction signals: 2,"{'OpenAI': 'Raqamli Gumanitar Fanlar', 'Claude': 'رقمی علوم انسانی', 'Gemini..."
265,kv,Komi,Uralic,TRANSMOGRIFICATION,5,False,False,Claude:borrowing; First Ollama:transliterat; Second Ollama:transliterat,Gemini:suffix,NaN,No Wikipedia; 5 distinct outputs; absence signals: 3; construction signals: 1,"{'Google Translate': 'Цифрӧвӧй гуманитарнӧй наукаяс', 'OpenAI': 'Цифровые гу..."
266,se,Northern Sami,Uralic,TRANSMOGRIFICATION,6,False,False,Claude:loanword; Gemini:loanword,Claude:compound; Claude:morpholog; Gemini:suffix,NaN,No Wikipedia; 6 distinct outputs; absence signals: 2; construction signals: 3,"{'Google Translate': 'Digitála humanisttalaš dieđat', 'OpenAI': 'Digiála hum..."
267,udm,Udmurt,Uralic,TRANSMOGRIFICATION,6,False,False,Claude:borrowing; Gemini:loanword,Claude:compound; Claude:construct; Gemini:suffix; First Ollama:suffix; Secon...,NaN,No Wikipedia; 6 distinct outputs; absence signals: 2; construction signals: 5,"{'Google Translate': 'Цифровой гуманитарной наукаос', 'OpenAI': 'Цифровыд гу..."


## 3.11 Build Explorer Data

Run the standalone script to merge disagreement analysis, confidence stats, and parsed service translations into `disagreement_explorer_data.csv` for the HTML explorer:

```bash
python scripts/exploration/build_disagreement_explorer_data.py
# optional flags:
#   --term "Digital Humanities"
#   --output-dir path/to/dir
```

Then open `html_files/disagreement_explorer.html` in a browser and load the CSV.